<a href="https://colab.research.google.com/github/van1403/TECHNOLOGY/blob/main/Technology_Solutions_Web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ TECHNOLOGY SOLUTIONS – Sistema web de servicio técnico

Este notebook levanta una **página web moderna** (Flask + SQLite + HTML/CSS/JS) desde Google Colab. Desde la página puedes hacer todo:

- **Tablero** tipo kanban con los equipos por estado y botón para avanzar cada orden
- **Nuevo servicio** con validaciones y una **orden de servicio imprimible**
- **Servicios**: buscar, cambiar estado y eliminar
- **Técnicos**: ver su carga de trabajo y agregar nuevos
- **Reportes** con gráficos y **descarga en Excel**

**Cómo usarlo:** ejecuta las celdas en orden. La celda 6 te da el enlace para abrir la página.

## 1. Configuración
Con `USAR_DRIVE = True` la base de datos se guarda en tu Google Drive y no se pierde al cerrar Colab.

In [1]:
!pip -q install flask openpyxl

import os

USAR_DRIVE = False  # Cambia a True para guardar los datos en Google Drive

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/TECHNOLOGY-SOLUTIONS'
else:
    CARPETA = '/content/TECHNOLOGY-SOLUTIONS'

os.makedirs(CARPETA, exist_ok=True)
os.makedirs('/content/ts_web/templates', exist_ok=True)
os.makedirs('/content/ts_web/static', exist_ok=True)
DB_PATH = os.path.join(CARPETA, 'technology_solutions.db')
print('📁 Base de datos:', DB_PATH)

📁 Base de datos: /content/TECHNOLOGY-SOLUTIONS/technology_solutions.db


## 2. Backend: API con Flask y SQLite
Crea las tablas `empleado`, `tecnico`, `cliente` y `servicio`, y expone la API que usa la página.

In [2]:
%%writefile /content/ts_web/app.py
import io
import os
import sqlite3
from datetime import date

import pandas as pd
from flask import Flask, jsonify, render_template, request, send_file

DB_PATH = os.environ.get('DB_PATH', 'technology_solutions.db')

TIPOS = ['Mantenimiento preventivo', 'Reparación', 'Formateo e instalación de software',
         'Instalación de redes', 'Recuperación de datos', 'Otro']
ESTADOS = ['Pendiente', 'En proceso', 'Terminado', 'Entregado']

app = Flask(__name__)


def conectar():
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    con.execute('PRAGMA foreign_keys = ON')
    return con


SQL_TABLAS = """
CREATE TABLE IF NOT EXISTS empleado (
    id_empleado INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre      TEXT NOT NULL,
    cargo       TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS tecnico (
    id_tecnico   INTEGER PRIMARY KEY AUTOINCREMENT,
    id_empleado  INTEGER NOT NULL UNIQUE,
    especialidad TEXT,
    FOREIGN KEY (id_empleado) REFERENCES empleado(id_empleado)
);
CREATE TABLE IF NOT EXISTS cliente (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    id_tecnico INTEGER,
    nombre     TEXT NOT NULL,
    dni        TEXT NOT NULL UNIQUE,
    celular    TEXT,
    FOREIGN KEY (id_tecnico) REFERENCES tecnico(id_tecnico)
);
CREATE TABLE IF NOT EXISTS servicio (
    id_servicio    INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente     INTEGER NOT NULL,
    tipo_servicio  TEXT NOT NULL,
    equipo         TEXT,
    descripcion    TEXT,
    monto          REAL NOT NULL CHECK (monto >= 0),
    fecha_servicio TEXT NOT NULL,
    estado         TEXT NOT NULL DEFAULT 'Pendiente',
    FOREIGN KEY (id_cliente) REFERENCES cliente(id_cliente) ON DELETE CASCADE
);
"""

SQL_SERVICIOS = """
SELECT s.id_servicio    AS id,
       s.fecha_servicio AS fecha,
       c.nombre         AS cliente,
       c.dni            AS dni,
       c.celular        AS celular,
       e.nombre         AS tecnico,
       s.tipo_servicio  AS tipo,
       s.equipo         AS equipo,
       s.descripcion    AS descripcion,
       s.monto          AS monto,
       s.estado         AS estado
FROM servicio s
JOIN cliente c       ON c.id_cliente  = s.id_cliente
LEFT JOIN tecnico t  ON t.id_tecnico  = c.id_tecnico
LEFT JOIN empleado e ON e.id_empleado = t.id_empleado
WHERE 1 = 1
"""


def iniciar_bd():
    with conectar() as con:
        con.executescript(SQL_TABLAS)
        if con.execute('SELECT COUNT(*) FROM empleado').fetchone()[0] == 0:
            con.executemany('INSERT INTO empleado (nombre, cargo) VALUES (?, ?)', [
                ('Carlos Ramírez', 'Técnico'), ('María Torres', 'Técnico'),
                ('Luis Paredes', 'Técnico'), ('Ana Quispe', 'Recepcionista'),
            ])
            con.executemany('INSERT INTO tecnico (id_empleado, especialidad) VALUES (?, ?)', [
                (1, 'Computadoras y laptops'), (2, 'Redes e internet'), (3, 'Impresoras y periféricos'),
            ])


def filas(cursor):
    return [dict(f) for f in cursor.fetchall()]


def error(mensaje, codigo=400):
    return jsonify(error=mensaje), codigo


@app.get('/')
def inicio():
    return render_template('index.html')


@app.get('/api/catalogos')
def catalogos():
    with conectar() as con:
        tecnicos = filas(con.execute('''
            SELECT t.id_tecnico AS id, e.nombre, t.especialidad
            FROM tecnico t JOIN empleado e ON e.id_empleado = t.id_empleado
            ORDER BY e.nombre'''))
    return jsonify(tipos=TIPOS, estados=ESTADOS, tecnicos=tecnicos)


@app.post('/api/tecnicos')
def crear_tecnico():
    d = request.get_json(silent=True) or {}
    nombre = str(d.get('nombre', '')).strip()
    especialidad = str(d.get('especialidad', '')).strip()
    if not nombre:
        return error('Escribe el nombre del técnico.')
    with conectar() as con:
        id_emp = con.execute('INSERT INTO empleado (nombre, cargo) VALUES (?, ?)',
                             (nombre, 'Técnico')).lastrowid
        con.execute('INSERT INTO tecnico (id_empleado, especialidad) VALUES (?, ?)',
                    (id_emp, especialidad or None))
    return jsonify(ok=True), 201


@app.get('/api/servicios')
def listar_servicios():
    with conectar() as con:
        return jsonify(filas(con.execute(SQL_SERVICIOS + ' ORDER BY s.id_servicio DESC')))


@app.post('/api/servicios')
def crear_servicio():
    d = request.get_json(silent=True) or {}
    nombre = str(d.get('nombre', '')).strip()
    dni = str(d.get('dni', '')).strip()
    celular = str(d.get('celular', '')).strip()
    tipo = d.get('tipo')
    equipo = str(d.get('equipo', '')).strip()
    descripcion = str(d.get('descripcion', '')).strip()
    fecha = str(d.get('fecha') or date.today().isoformat())
    try:
        monto = round(float(d.get('monto') or 0), 2)
    except (TypeError, ValueError):
        monto = -1
    try:
        id_tecnico = int(d.get('id_tecnico'))
    except (TypeError, ValueError):
        return error('Elige el técnico que atenderá el servicio.')

    if not nombre:
        return error('Escribe el nombre del cliente.')
    if not (dni.isdigit() and len(dni) == 8):
        return error('El DNI debe tener exactamente 8 dígitos.')
    if celular and not (celular.isdigit() and len(celular) == 9):
        return error('El celular debe tener 9 dígitos.')
    if tipo not in TIPOS:
        return error('Elige un tipo de servicio de la lista.')
    if monto <= 0:
        return error('El monto debe ser mayor a 0.')

    with conectar() as con:
        if not con.execute('SELECT 1 FROM tecnico WHERE id_tecnico = ?', (id_tecnico,)).fetchone():
            return error('El técnico elegido no existe.')
        fila = con.execute('SELECT id_cliente FROM cliente WHERE dni = ?', (dni,)).fetchone()
        if fila:
            id_cliente = fila['id_cliente']
            con.execute('UPDATE cliente SET nombre = ?, celular = ?, id_tecnico = ? WHERE id_cliente = ?',
                        (nombre, celular, id_tecnico, id_cliente))
        else:
            id_cliente = con.execute(
                'INSERT INTO cliente (id_tecnico, nombre, dni, celular) VALUES (?, ?, ?, ?)',
                (id_tecnico, nombre, dni, celular)).lastrowid
        id_servicio = con.execute('''
            INSERT INTO servicio (id_cliente, tipo_servicio, equipo, descripcion, monto, fecha_servicio, estado)
            VALUES (?, ?, ?, ?, ?, ?, 'Pendiente')''',
            (id_cliente, tipo, equipo, descripcion, monto, fecha)).lastrowid
        servicio = dict(con.execute(SQL_SERVICIOS + ' AND s.id_servicio = ?', (id_servicio,)).fetchone())
    return jsonify(servicio=servicio), 201


@app.patch('/api/servicios/<int:id_servicio>')
def cambiar_estado(id_servicio):
    estado = (request.get_json(silent=True) or {}).get('estado')
    if estado not in ESTADOS:
        return error('Estado no válido.')
    with conectar() as con:
        n = con.execute('UPDATE servicio SET estado = ? WHERE id_servicio = ?', (estado, id_servicio)).rowcount
    return jsonify(ok=True) if n else error('No existe un servicio con ese número.', 404)


@app.delete('/api/servicios/<int:id_servicio>')
def eliminar_servicio(id_servicio):
    with conectar() as con:
        n = con.execute('DELETE FROM servicio WHERE id_servicio = ?', (id_servicio,)).rowcount
    return jsonify(ok=True) if n else error('No existe un servicio con ese número.', 404)


@app.get('/api/resumen')
def resumen():
    with conectar() as con:
        tot = con.execute('SELECT COUNT(*) AS total, COALESCE(SUM(monto), 0) AS ingresos FROM servicio').fetchone()
        por_estado = {e: 0 for e in ESTADOS}
        for f in con.execute('SELECT estado, COUNT(*) AS n FROM servicio GROUP BY estado'):
            por_estado[f['estado']] = f['n']
        base = '''FROM servicio s
                  JOIN cliente c       ON c.id_cliente  = s.id_cliente
                  LEFT JOIN tecnico t  ON t.id_tecnico  = c.id_tecnico
                  LEFT JOIN empleado e ON e.id_empleado = t.id_empleado'''
        por_tecnico = filas(con.execute(f'''
            SELECT COALESCE(e.nombre, 'Sin técnico') AS tecnico, COUNT(*) AS cantidad, SUM(s.monto) AS ingresos
            {base} GROUP BY 1 ORDER BY ingresos DESC'''))
        por_tipo = filas(con.execute('''
            SELECT tipo_servicio AS tipo, COUNT(*) AS cantidad, SUM(monto) AS ingresos
            FROM servicio GROUP BY 1 ORDER BY cantidad DESC'''))
    return jsonify(total=tot['total'], ingresos=tot['ingresos'], por_estado=por_estado,
                   por_tecnico=por_tecnico, por_tipo=por_tipo)


@app.get('/api/exportar')
def exportar():
    with conectar() as con:
        df = pd.read_sql_query(SQL_SERVICIOS + ' ORDER BY s.id_servicio', con)
    df.columns = ['N°', 'Fecha', 'Cliente', 'DNI', 'Celular', 'Técnico', 'Servicio',
                  'Equipo', 'Descripción', 'Monto (S/)', 'Estado']
    buffer = io.BytesIO()
    df.to_excel(buffer, index=False, sheet_name='Servicios')
    buffer.seek(0)
    return send_file(buffer, as_attachment=True, download_name='servicios_technology_solutions.xlsx',
                     mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')


iniciar_bd()

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=int(os.environ.get('PORT', 5000)))

Writing /content/ts_web/app.py


## 3. Página principal (HTML)

In [3]:
%%writefile /content/ts_web/templates/index.html
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Technology Solutions | Servicio técnico</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600&family=Sora:wght@500;600;700&display=swap" rel="stylesheet">
  <link rel="stylesheet" href="{{ url_for('static', filename='styles.css') }}">
</head>
<body>
<div class="app">
  <aside class="sidebar">
    <div class="brand">
      <span class="brand-mark" aria-hidden="true">TS</span>
      <div><strong>Technology Solutions</strong><small>Servicio técnico</small></div>
    </div>
    <nav aria-label="Secciones">
      <button class="nav-btn active" data-view="tablero" aria-current="page">Tablero</button>
      <button class="nav-btn" data-view="nuevo">Nuevo servicio</button>
      <button class="nav-btn" data-view="servicios">Servicios</button>
      <button class="nav-btn" data-view="tecnicos">Técnicos</button>
      <button class="nav-btn" data-view="reportes">Reportes</button>
    </nav>
  </aside>

  <main>
    <!-- TABLERO -->
    <section id="view-tablero" class="view active">
      <header class="view-head">
        <div>
          <h1>Tablero de trabajo</h1>
          <p>Los equipos del taller según su estado. Avanza cada orden cuando cambie.</p>
        </div>
        <button class="btn primary" data-view="nuevo">Nuevo servicio</button>
      </header>
      <div class="stats" id="stats"></div>
      <div class="board" id="board"></div>
    </section>

    <!-- NUEVO SERVICIO -->
    <section id="view-nuevo" class="view">
      <header class="view-head">
        <div>
          <h1>Nuevo servicio</h1>
          <p>Registra al cliente y el equipo que deja en el taller.</p>
        </div>
      </header>
      <div class="nuevo-grid">
        <form id="form-servicio" class="panel" novalidate>
          <fieldset>
            <legend>Cliente</legend>
            <label>Nombre completo<input name="nombre" autocomplete="name" required></label>
            <div class="row">
              <label>DNI<input name="dni" inputmode="numeric" maxlength="8" placeholder="8 dígitos" required></label>
              <label>Celular<input name="celular" inputmode="numeric" maxlength="9" placeholder="9 dígitos (opcional)"></label>
            </div>
          </fieldset>
          <fieldset>
            <legend>Equipo y trabajo</legend>
            <div class="row">
              <label>Tipo de servicio<select name="tipo"></select></label>
              <label>Técnico<select name="id_tecnico"></select></label>
            </div>
            <label>Equipo<input name="equipo" placeholder="Ej. Laptop HP 15"></label>
            <label>Falla reportada o trabajo a realizar<textarea name="descripcion" rows="3"></textarea></label>
            <div class="row">
              <label>Monto (S/)<input name="monto" type="number" min="0" step="0.5" placeholder="0.00"></label>
              <label>Fecha<input name="fecha" type="date"></label>
            </div>
          </fieldset>
          <p class="form-error" id="form-error" role="alert"></p>
          <button class="btn primary" type="submit">Registrar servicio</button>
        </form>
        <div class="ticket-wrap" id="ticket-wrap">
          <div class="ticket-empty">La orden de servicio aparecerá aquí cuando registres uno, lista para imprimir y entregar al cliente.</div>
        </div>
      </div>
    </section>

    <!-- SERVICIOS -->
    <section id="view-servicios" class="view">
      <header class="view-head">
        <div>
          <h1>Servicios</h1>
          <p>Busca por cliente, DNI o número de orden. Cambia el estado desde la tabla.</p>
        </div>
      </header>
      <div class="toolbar">
        <input id="filtro-texto" type="search" placeholder="Buscar cliente, DNI o N°" aria-label="Buscar servicios">
        <select id="filtro-estado" aria-label="Filtrar por estado"></select>
        <span class="toolbar-info" id="tabla-info"></span>
      </div>
      <div class="panel table-wrap">
        <table>
          <thead>
            <tr><th>N°</th><th>Fecha</th><th>Cliente</th><th>Servicio</th><th>Técnico</th><th class="t-num">Monto</th><th>Estado</th><th><span class="sr-only">Acciones</span></th></tr>
          </thead>
          <tbody id="tabla"></tbody>
        </table>
      </div>
    </section>

    <!-- TÉCNICOS -->
    <section id="view-tecnicos" class="view">
      <header class="view-head">
        <div>
          <h1>Técnicos</h1>
          <p>El equipo que atiende los servicios y cuántos trabajos tiene en curso.</p>
        </div>
      </header>
      <div class="tec-grid">
        <div class="panel"><ul class="lista-tec" id="lista-tecnicos"></ul></div>
        <form id="form-tecnico" class="panel" novalidate>
          <fieldset>
            <legend>Agregar técnico</legend>
            <label>Nombre completo<input name="nombre" required></label>
            <label>Especialidad<input name="especialidad" placeholder="Ej. Celulares y tablets"></label>
          </fieldset>
          <button class="btn primary" type="submit">Agregar técnico</button>
        </form>
      </div>
    </section>

    <!-- REPORTES -->
    <section id="view-reportes" class="view">
      <header class="view-head">
        <div>
          <h1>Reportes</h1>
          <p>Ingresos y carga de trabajo de todos los servicios registrados.</p>
        </div>
        <a class="btn primary" href="api/exportar" download>Descargar Excel</a>
      </header>
      <div class="stats" id="rep-stats"></div>
      <div id="rep-contenido">
        <div class="rep-graficos">
          <div class="panel"><h2>Ingresos por técnico</h2><div class="grafico"><canvas id="g-tecnicos" aria-label="Ingresos por técnico"></canvas></div></div>
          <div class="panel"><h2>Servicios por estado</h2><div class="grafico"><canvas id="g-estados" aria-label="Servicios por estado"></canvas></div></div>
        </div>
        <div class="panel table-wrap">
          <table class="tabla-simple">
            <thead><tr><th>Tipo de servicio</th><th class="t-num">Cantidad</th><th class="t-num">Ingresos</th></tr></thead>
            <tbody id="rep-tipos"></tbody>
          </table>
        </div>
      </div>
    </section>
  </main>
</div>

<div id="toast" class="toast" role="status" aria-live="polite"></div>

<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<script src="{{ url_for('static', filename='app.js') }}"></script>
</body>
</html>

Writing /content/ts_web/templates/index.html


## 4. Diseño (CSS)
Incluye modo oscuro automático, diseño adaptable a celular y estilo de impresión para la orden.

In [4]:
%%writefile /content/ts_web/static/styles.css
:root {
  --bg: #EDF0F5;
  --surface: #FFFFFF;
  --surface-2: #F6F8FB;
  --ink: #15213B;
  --muted: #5A6681;
  --line: #D9DFE9;
  --accent: #2C59E6;
  --accent-soft: #E3EAFD;
  --danger: #C8372D;
  --pendiente: #B97300;
  --proceso: #2C59E6;
  --terminado: #16875F;
  --entregado: #6E7B94;
  --nav-bg: #15213B;
  --nav-ink: #B8C3DA;
  --sombra: 0 1px 2px rgba(21, 33, 59, .06), 0 8px 24px rgba(21, 33, 59, .08);
  --fuente-titulo: 'Sora', 'Segoe UI', system-ui, sans-serif;
  --fuente: 'IBM Plex Sans', 'Segoe UI', system-ui, sans-serif;
  color-scheme: light;
}

@media (prefers-color-scheme: dark) {
  :root {
    --bg: #0D1424;
    --surface: #151F34;
    --surface-2: #1A2640;
    --ink: #E6EBF5;
    --muted: #98A4BD;
    --line: #283551;
    --accent: #6A8CF7;
    --accent-soft: #1F2E57;
    --danger: #F0776C;
    --pendiente: #F0B23E;
    --proceso: #6A8CF7;
    --terminado: #3CC495;
    --entregado: #8C98AF;
    --nav-bg: #090F1C;
    --sombra: 0 8px 24px rgba(0, 0, 0, .35);
    color-scheme: dark;
  }
}

* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--ink); font: 15px/1.5 var(--fuente); }
h1, h2, h3 { font-family: var(--fuente-titulo); margin: 0; line-height: 1.2; }
.sr-only { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); }

.pendiente { --c: var(--pendiente); }
.proceso   { --c: var(--proceso); }
.terminado { --c: var(--terminado); }
.entregado { --c: var(--entregado); }

/* Estructura */
.app { display: grid; grid-template-columns: 240px 1fr; min-height: 100vh; }
.sidebar {
  background: var(--nav-bg); color: var(--nav-ink); padding: 24px 16px;
  position: sticky; top: 0; height: 100vh; display: flex; flex-direction: column; gap: 32px;
}
.brand { display: flex; gap: 12px; align-items: center; color: #fff; }
.brand-mark {
  width: 40px; height: 40px; border-radius: 10px; background: var(--accent); flex: none;
  display: grid; place-items: center; font: 700 15px var(--fuente-titulo); color: #fff;
}
.brand strong { display: block; font: 600 15px var(--fuente-titulo); }
.brand small { color: var(--nav-ink); font-size: 12.5px; }
nav { display: flex; flex-direction: column; gap: 4px; }
.nav-btn {
  all: unset; cursor: pointer; padding: 10px 14px; border-radius: 8px;
  color: var(--nav-ink); font-weight: 500;
}
.nav-btn:hover { background: rgba(255, 255, 255, .06); color: #fff; }
.nav-btn.active { background: rgba(255, 255, 255, .1); color: #fff; box-shadow: inset 3px 0 0 var(--accent); }

main { padding: 32px clamp(16px, 4vw, 48px) 64px; min-width: 0; max-width: 1440px; }
.view { display: none; }
.view.active { display: block; animation: entrar .2s ease-out; }
@keyframes entrar { from { opacity: 0; } }

.view-head { display: flex; justify-content: space-between; align-items: flex-end; gap: 16px; flex-wrap: wrap; margin-bottom: 24px; }
.view-head h1 { font-size: 28px; font-weight: 700; letter-spacing: -.01em; }
.view-head p { margin: 4px 0 0; color: var(--muted); max-width: 60ch; }

/* Controles */
.btn {
  font: 600 14px/1 var(--fuente); border: 1px solid var(--line); background: var(--surface);
  color: var(--ink); padding: 11px 16px; border-radius: 8px; cursor: pointer;
  text-decoration: none; display: inline-flex; align-items: center; gap: 8px;
}
.btn:hover { border-color: var(--muted); }
.btn.primary { background: var(--accent); border-color: var(--accent); color: #fff; }
.btn.primary:hover { filter: brightness(1.08); }
.btn.small { padding: 7px 10px; font-size: 13px; }
.btn.ghost { background: transparent; }
.btn.danger { color: var(--danger); }
.btn:disabled { opacity: .6; cursor: progress; }

:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }

.panel { background: var(--surface); border: 1px solid var(--line); border-radius: 12px; padding: 24px; }
.panel h2 { font-size: 16px; font-weight: 600; margin-bottom: 16px; }

fieldset { border: 0; padding: 0; margin: 0 0 12px; }
legend { font: 600 16px var(--fuente-titulo); margin-bottom: 14px; padding: 0; }
label { display: flex; flex-direction: column; gap: 6px; font-size: 13.5px; font-weight: 500; color: var(--muted); margin-bottom: 14px; }
input, select, textarea {
  font: 15px var(--fuente); color: var(--ink); background: var(--surface-2);
  border: 1px solid var(--line); border-radius: 8px; padding: 10px 12px; width: 100%;
}
textarea { resize: vertical; }
input:focus, select:focus, textarea:focus { outline: 2px solid var(--accent); outline-offset: -1px; border-color: var(--accent); }
.row { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
.form-error { color: var(--danger); min-height: 1.4em; margin: 0 0 12px; font-weight: 500; }

/* Indicadores */
.stats {
  display: grid; grid-template-columns: repeat(4, 1fr); background: var(--surface);
  border: 1px solid var(--line); border-radius: 12px; margin-bottom: 24px;
}
.stat { padding: 16px 20px; border-left: 1px solid var(--line); }
.stat:first-child { border-left: 0; }
.stat-valor { display: block; font: 600 26px var(--fuente-titulo); font-variant-numeric: tabular-nums; }
.stat-label { color: var(--muted); font-size: 13px; }

/* Tablero */
.board { display: grid; grid-template-columns: repeat(4, minmax(220px, 1fr)); gap: 16px; overflow-x: auto; padding-bottom: 8px; }
.col { background: var(--surface-2); border: 1px solid var(--line); border-top: 4px solid var(--c); border-radius: 12px; min-height: 320px; }
.col-head { display: flex; justify-content: space-between; align-items: center; padding: 14px 16px 10px; }
.col-head h2 { font-size: 15px; font-weight: 600; }
.col-count {
  background: color-mix(in srgb, var(--c) 15%, transparent); color: var(--c);
  font-weight: 600; font-size: 13px; border-radius: 999px; padding: 2px 10px;
}
.col-body { padding: 0 12px 12px; display: flex; flex-direction: column; gap: 10px; }
.col-vacia { color: var(--muted); font-size: 13.5px; margin: 4px; }
.col-mas { color: var(--muted); font-size: 13px; margin: 2px 4px; }
.card { background: var(--surface); border: 1px solid var(--line); border-radius: 8px; padding: 12px 14px; }
.card-top { display: flex; justify-content: space-between; font-size: 12.5px; color: var(--muted); }
.card-monto { font-weight: 600; color: var(--ink); font-variant-numeric: tabular-nums; }
.card h3 { font-size: 15px; font-weight: 600; margin: 6px 0 2px; }
.card-cliente { margin: 0; font-weight: 500; }
.card-meta { margin: 6px 0 0; color: var(--muted); font-size: 13px; }
.card .btn { margin-top: 10px; width: 100%; justify-content: center; }

/* Nuevo servicio y orden */
.nuevo-grid { display: grid; grid-template-columns: minmax(0, 1.3fr) minmax(300px, 1fr); gap: 24px; align-items: start; }
.ticket-wrap { position: sticky; top: 24px; }
.ticket-empty { border: 2px dashed var(--line); border-radius: 12px; padding: 56px 28px; text-align: center; color: var(--muted); }
.ticket {
  background: var(--surface); border: 1px solid var(--line); border-radius: 12px;
  padding: 24px; box-shadow: var(--sombra); animation: imprimir .6s cubic-bezier(.2, .7, .2, 1);
}
@keyframes imprimir {
  from { clip-path: inset(0 -40px 100% -40px); transform: translateY(-12px); }
  to   { clip-path: inset(0 -40px -20px -40px); transform: none; }
}
.ticket-head {
  display: flex; justify-content: space-between; align-items: flex-start; gap: 12px;
  padding-bottom: 18px; margin-bottom: 18px; border-bottom: 2px dashed var(--line); position: relative;
}
.ticket-head::before, .ticket-head::after {
  content: ''; position: absolute; bottom: -11px; width: 20px; height: 20px;
  border-radius: 50%; background: var(--bg);
}
.ticket-head::before { left: -35px; }
.ticket-head::after { right: -35px; }
.ticket-head strong { font: 600 15px var(--fuente-titulo); }
.ticket-head div span { display: block; color: var(--muted); font-size: 13px; }
.ticket-nro { font: 700 24px var(--fuente-titulo); color: var(--accent); white-space: nowrap; }
.ticket dl { display: grid; grid-template-columns: 1fr 1fr; gap: 12px 16px; margin: 0; }
.ticket dt { font-size: 12px; color: var(--muted); }
.ticket dd { margin: 0; font-weight: 500; overflow-wrap: anywhere; }
.ticket .full { grid-column: 1 / -1; }
.ticket-total {
  display: flex; justify-content: space-between; align-items: baseline;
  margin-top: 18px; padding-top: 14px; border-top: 2px dashed var(--line);
}
.ticket-total strong { font: 700 26px var(--fuente-titulo); font-variant-numeric: tabular-nums; }
.ticket-pie { margin: 14px 0 0; font-size: 12.5px; color: var(--muted); text-align: center; }
.ticket-acciones { display: flex; gap: 10px; margin-top: 14px; flex-wrap: wrap; }

/* Tabla */
.toolbar { display: flex; gap: 12px; margin-bottom: 16px; flex-wrap: wrap; align-items: center; }
.toolbar input { flex: 1 1 260px; max-width: 420px; }
.toolbar select { width: auto; }
.toolbar-info { color: var(--muted); font-size: 13.5px; margin-left: auto; }
.table-wrap { padding: 0; overflow-x: auto; }
table { width: 100%; border-collapse: collapse; font-size: 14px; }
#view-servicios table { min-width: 880px; }
th { text-align: left; font-weight: 600; font-size: 13px; color: var(--muted); padding: 12px 16px; border-bottom: 1px solid var(--line); background: var(--surface-2); }
td { padding: 12px 16px; border-bottom: 1px solid var(--line); vertical-align: top; }
td small { display: block; color: var(--muted); font-size: 12.5px; }
tbody tr:last-child td { border-bottom: 0; }
.t-nro { font-weight: 600; white-space: nowrap; }
.t-fecha { white-space: nowrap; }
.t-num { text-align: right; white-space: nowrap; font-variant-numeric: tabular-nums; }
.t-vacia { text-align: center; color: var(--muted); padding: 40px 16px; }
.estado-select {
  width: auto; padding: 6px 10px; font-size: 13px; font-weight: 600; border-radius: 999px;
  color: var(--c); background: color-mix(in srgb, var(--c) 12%, var(--surface));
  border-color: color-mix(in srgb, var(--c) 35%, transparent);
}

/* Técnicos */
.tec-grid { display: grid; grid-template-columns: minmax(0, 1.4fr) minmax(280px, 1fr); gap: 24px; align-items: start; }
.lista-tec { list-style: none; margin: -8px 0; padding: 0; }
.tec { display: flex; align-items: center; gap: 14px; padding: 14px 0; border-bottom: 1px solid var(--line); }
.tec:last-child { border-bottom: 0; }
.tec-ini {
  width: 40px; height: 40px; border-radius: 50%; flex: none; background: var(--accent-soft);
  color: var(--accent); display: grid; place-items: center; font-weight: 600;
}
.tec div { flex: 1; min-width: 0; }
.tec small { display: block; color: var(--muted); }
.tec-carga { font-size: 13px; color: var(--muted); white-space: nowrap; }

/* Reportes */
.rep-graficos { display: grid; grid-template-columns: minmax(0, 1.4fr) minmax(0, 1fr); gap: 24px; margin-bottom: 24px; }
.grafico { position: relative; height: 280px; }
.rep-vacio { text-align: center; color: var(--muted); padding: 48px 24px; }

/* Aviso */
.toast {
  position: fixed; right: 24px; bottom: 24px; max-width: 380px; z-index: 20;
  background: var(--ink); color: var(--bg); padding: 12px 18px; border-radius: 10px; font-weight: 500;
  opacity: 0; transform: translateY(16px); pointer-events: none; transition: opacity .2s, transform .2s;
}
.toast.show { opacity: 1; transform: none; }
.toast.error { background: var(--danger); color: #fff; }

/* Adaptable */
@media (max-width: 1100px) {
  .board { grid-template-columns: repeat(4, 250px); }
  .nuevo-grid, .tec-grid, .rep-graficos { grid-template-columns: 1fr; }
  .ticket-wrap { position: static; }
  .stats { grid-template-columns: repeat(2, 1fr); }
  .stat:nth-child(3) { border-left: 0; }
  .stat:nth-child(n + 3) { border-top: 1px solid var(--line); }
}
@media (max-width: 760px) {
  .app { grid-template-columns: 1fr; }
  .sidebar { height: auto; flex-direction: row; align-items: center; gap: 16px; padding: 12px 16px; overflow-x: auto; z-index: 10; }
  .brand small { display: none; }
  .brand strong { white-space: nowrap; }
  nav { flex-direction: row; }
  .nav-btn { white-space: nowrap; }
  .nav-btn.active { box-shadow: inset 0 -3px 0 var(--accent); }
  .row { grid-template-columns: 1fr; }
  main { padding: 20px 16px 48px; }
  .view-head h1 { font-size: 24px; }
}
@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { animation: none !important; transition: none !important; }
}
@media print {
  body { background: #fff; }
  .sidebar, .view-head, form, .ticket-acciones, .toast { display: none !important; }
  .app, .nuevo-grid { display: block; }
  main { padding: 0; }
  .ticket { box-shadow: none; border: 1px solid #999; max-width: 420px; animation: none; }
  .ticket-head::before, .ticket-head::after { display: none; }
}

Writing /content/ts_web/static/styles.css


## 5. Funcionamiento de la página (JavaScript)

In [5]:
%%writefile /content/ts_web/static/app.js
const $ = (sel, el = document) => el.querySelector(sel);
const $$ = (sel, el = document) => [...el.querySelectorAll(sel)];

const CLASE_ESTADO = { 'Pendiente': 'pendiente', 'En proceso': 'proceso', 'Terminado': 'terminado', 'Entregado': 'entregado' };
const ACCION_SIGUIENTE = { 'Pendiente': 'Empezar trabajo', 'En proceso': 'Marcar como terminado', 'Terminado': 'Entregar al cliente' };

let catalogos = { tipos: [], estados: [], tecnicos: [] };
let servicios = [];
const graficos = {};

const soles = n => 'S/ ' + Number(n || 0).toLocaleString('es-PE', { minimumFractionDigits: 2, maximumFractionDigits: 2 });
const nro = id => 'N° ' + String(id).padStart(5, '0');
const esc = v => String(v ?? '').replace(/[&<>"']/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;' }[c]));
const iniciales = nombre => nombre.split(/\s+/).filter(Boolean).slice(0, 2).map(p => p[0].toUpperCase()).join('');
const hoy = () => {
  const d = new Date();
  d.setMinutes(d.getMinutes() - d.getTimezoneOffset());
  return d.toISOString().slice(0, 10);
};
const fechaLegible = f => f ? new Date(f + 'T00:00:00').toLocaleDateString('es-PE', { day: '2-digit', month: 'short', year: 'numeric' }) : '';

async function api(ruta, opciones = {}) {
  const res = await fetch(ruta, { headers: { 'Content-Type': 'application/json' }, ...opciones });
  const datos = await res.json().catch(() => ({}));
  if (!res.ok) throw new Error(datos.error || `El servidor respondió con el error ${res.status}.`);
  return datos;
}

let temporizador;
function aviso(mensaje, tipo = 'ok') {
  const t = $('#toast');
  t.textContent = mensaje;
  t.className = `toast show ${tipo}`;
  clearTimeout(temporizador);
  temporizador = setTimeout(() => t.classList.remove('show'), 3500);
}

function mostrarVista(nombre) {
  $$('.view').forEach(v => v.classList.toggle('active', v.id === `view-${nombre}`));
  $$('.nav-btn').forEach(b => {
    const activo = b.dataset.view === nombre;
    b.classList.toggle('active', activo);
    if (activo) b.setAttribute('aria-current', 'page'); else b.removeAttribute('aria-current');
  });
  if (nombre === 'reportes') renderReportes();
  if (nombre === 'nuevo') $('#form-servicio').nombre.focus();
  window.scrollTo(0, 0);
}

/* ---------- Carga de datos ---------- */
async function cargarCatalogos() {
  catalogos = await api('api/catalogos');
  const f = $('#form-servicio');
  f.tipo.innerHTML = catalogos.tipos.map(t => `<option>${esc(t)}</option>`).join('');
  f.id_tecnico.innerHTML = catalogos.tecnicos
    .map(t => `<option value="${t.id}">${esc(t.nombre)}${t.especialidad ? ' – ' + esc(t.especialidad) : ''}</option>`).join('');
  $('#filtro-estado').innerHTML = '<option value="">Todos los estados</option>' +
    catalogos.estados.map(e => `<option>${esc(e)}</option>`).join('');
}

async function cargarServicios() {
  servicios = await api('api/servicios');
  renderTablero();
  renderTabla();
  renderTecnicos();
}

/* ---------- Tablero ---------- */
function renderTablero() {
  const cuenta = e => servicios.filter(s => s.estado === e).length;
  const enTaller = servicios.filter(s => s.estado !== 'Entregado').length;
  const porCobrar = servicios.filter(s => s.estado === 'Terminado').reduce((a, s) => a + s.monto, 0);
  $('#stats').innerHTML = [
    [enTaller, 'Equipos en el taller'],
    [cuenta('Pendiente'), 'Por empezar'],
    [cuenta('Terminado'), 'Listos para entregar'],
    [soles(porCobrar), 'Por cobrar al entregar'],
  ].map(([v, l]) => `<div class="stat"><span class="stat-valor">${v}</span><span class="stat-label">${l}</span></div>`).join('');

  $('#board').innerHTML = catalogos.estados.map(estado => {
    let lista = servicios.filter(s => s.estado === estado);
    let extra = '';
    if (estado === 'Entregado' && lista.length > 5) {
      extra = `<p class="col-mas">y ${lista.length - 5} más en Servicios</p>`;
      lista = lista.slice(0, 5);
    }
    const cuerpo = lista.length ? lista.map(tarjeta).join('') + extra
      : `<p class="col-vacia">${estado === 'Pendiente' ? 'No hay equipos esperando. Registra un nuevo servicio para empezar.' : 'Sin equipos en este estado.'}</p>`;
    return `<section class="col ${CLASE_ESTADO[estado]}">
      <header class="col-head"><h2>${estado}</h2><span class="col-count">${cuenta(estado)}</span></header>
      <div class="col-body">${cuerpo}</div>
    </section>`;
  }).join('');
}

function tarjeta(s) {
  const siguiente = ACCION_SIGUIENTE[s.estado];
  return `<article class="card">
    <div class="card-top"><span>${nro(s.id)}</span><span class="card-monto">${soles(s.monto)}</span></div>
    <h3>${esc(s.equipo || s.tipo)}</h3>
    <p class="card-cliente">${esc(s.cliente)}</p>
    <p class="card-meta">${esc(s.tipo)}<br>${esc(s.tecnico || 'Sin técnico')}</p>
    ${siguiente ? `<button class="btn small" data-avanzar="${s.id}">${siguiente}</button>` : ''}
  </article>`;
}

async function cambiarEstado(id, estado) {
  try {
    await api(`api/servicios/${id}`, { method: 'PATCH', body: JSON.stringify({ estado }) });
    aviso(`${nro(id)} pasó a «${estado}».`);
  } catch (e) {
    aviso(e.message, 'error');
  }
  await cargarServicios();
}

/* ---------- Tabla de servicios ---------- */
function renderTabla() {
  const q = $('#filtro-texto').value.trim().toLowerCase();
  const est = $('#filtro-estado').value;
  const lista = servicios.filter(s =>
    (!est || s.estado === est) &&
    (!q || s.cliente.toLowerCase().includes(q) || s.dni.includes(q) || String(s.id).includes(q)));
  $('#tabla-info').textContent = `${lista.length} de ${servicios.length} servicios`;
  $('#tabla').innerHTML = lista.length ? lista.map(s => `<tr>
      <td class="t-nro">${nro(s.id)}</td>
      <td class="t-fecha">${fechaLegible(s.fecha)}</td>
      <td><strong>${esc(s.cliente)}</strong><small>DNI ${esc(s.dni)}${s.celular ? ' | Cel. ' + esc(s.celular) : ''}</small></td>
      <td>${esc(s.tipo)}<small>${esc(s.equipo)}</small></td>
      <td>${esc(s.tecnico || '—')}</td>
      <td class="t-num">${soles(s.monto)}</td>
      <td><select class="estado-select ${CLASE_ESTADO[s.estado]}" data-estado-id="${s.id}" aria-label="Estado de la orden ${s.id}">
        ${catalogos.estados.map(e => `<option ${e === s.estado ? 'selected' : ''}>${e}</option>`).join('')}
      </select></td>
      <td><button class="btn ghost danger small" data-eliminar="${s.id}">Eliminar</button></td>
    </tr>`).join('')
    : `<tr><td colspan="8" class="t-vacia">${servicios.length ? 'Ningún servicio coincide con la búsqueda. Prueba con otro nombre, DNI o estado.' : 'Todavía no hay servicios. Regístralos desde Nuevo servicio.'}</td></tr>`;
}

async function eliminar(id) {
  if (!confirm(`¿Eliminar la orden ${nro(id)}? Esta acción no se puede deshacer.`)) return;
  try {
    await api(`api/servicios/${id}`, { method: 'DELETE' });
    aviso(`Orden ${nro(id)} eliminada.`);
  } catch (e) {
    aviso(e.message, 'error');
  }
  await cargarServicios();
}

/* ---------- Nuevo servicio ---------- */
function mostrarOrden(s) {
  const campo = (titulo, valor, completo = false) => valor
    ? `<div${completo ? ' class="full"' : ''}><dt>${titulo}</dt><dd>${esc(valor)}</dd></div>` : '';
  $('#ticket-wrap').innerHTML = `<article class="ticket">
      <header class="ticket-head">
        <div><strong>Technology Solutions</strong><span>Orden de servicio técnico</span></div>
        <span class="ticket-nro">${nro(s.id)}</span>
      </header>
      <dl>
        ${campo('Fecha', fechaLegible(s.fecha))}
        ${campo('Estado', s.estado)}
        ${campo('Cliente', s.cliente, true)}
        ${campo('DNI', s.dni)}
        ${campo('Celular', s.celular)}
        ${campo('Servicio', s.tipo)}
        ${campo('Equipo', s.equipo)}
        ${campo('Falla reportada', s.descripcion, true)}
        ${campo('Técnico asignado', s.tecnico, true)}
      </dl>
      <div class="ticket-total"><span>Total</span><strong>${soles(s.monto)}</strong></div>
      <p class="ticket-pie">Presente esta orden al recoger su equipo.</p>
    </article>
    <div class="ticket-acciones">
      <button class="btn" data-imprimir>Imprimir orden</button>
      <button class="btn ghost" data-view="tablero">Ver en el tablero</button>
    </div>`;
}

$('#form-servicio').addEventListener('submit', async e => {
  e.preventDefault();
  const f = e.currentTarget;
  const boton = f.querySelector('button[type=submit]');
  $('#form-error').textContent = '';
  boton.disabled = true;
  try {
    const { servicio } = await api('api/servicios', { method: 'POST', body: JSON.stringify(Object.fromEntries(new FormData(f))) });
    mostrarOrden(servicio);
    f.reset();
    f.fecha.value = hoy();
    aviso(`Servicio ${nro(servicio.id)} registrado.`);
    await cargarServicios();
  } catch (err) {
    $('#form-error').textContent = err.message;
  } finally {
    boton.disabled = false;
  }
});

/* ---------- Técnicos ---------- */
function renderTecnicos() {
  $('#lista-tecnicos').innerHTML = catalogos.tecnicos.map(t => {
    const enCurso = servicios.filter(s => s.tecnico === t.nombre && s.estado !== 'Entregado').length;
    return `<li class="tec">
      <span class="tec-ini" aria-hidden="true">${esc(iniciales(t.nombre))}</span>
      <div><strong>${esc(t.nombre)}</strong><small>${esc(t.especialidad || 'Sin especialidad registrada')}</small></div>
      <span class="tec-carga">${enCurso} en curso</span>
    </li>`;
  }).join('');
}

$('#form-tecnico').addEventListener('submit', async e => {
  e.preventDefault();
  const f = e.currentTarget;
  try {
    await api('api/tecnicos', { method: 'POST', body: JSON.stringify(Object.fromEntries(new FormData(f))) });
    aviso(`Técnico ${f.nombre.value.trim()} agregado.`);
    f.reset();
    await cargarCatalogos();
    renderTecnicos();
  } catch (err) {
    aviso(err.message, 'error');
  }
});

/* ---------- Reportes ---------- */
async function renderReportes() {
  let r;
  try {
    r = await api('api/resumen');
  } catch (e) {
    aviso(e.message, 'error');
    return;
  }
  const entregados = r.por_estado['Entregado'] || 0;
  $('#rep-stats').innerHTML = [
    [r.total, 'Servicios registrados'],
    [soles(r.ingresos), 'Ingresos totales'],
    [soles(r.total ? r.ingresos / r.total : 0), 'Promedio por servicio'],
    [entregados, 'Equipos entregados'],
  ].map(([v, l]) => `<div class="stat"><span class="stat-valor">${v}</span><span class="stat-label">${l}</span></div>`).join('');

  const contenido = $('#rep-contenido');
  if (!r.total) {
    contenido.hidden = true;
    if (!$('#rep-vacio')) contenido.insertAdjacentHTML('beforebegin', '<div class="panel rep-vacio" id="rep-vacio">Los gráficos aparecerán cuando registres el primer servicio.</div>');
    return;
  }
  contenido.hidden = false;
  $('#rep-vacio')?.remove();

  $('#rep-tipos').innerHTML = r.por_tipo.map(t =>
    `<tr><td>${esc(t.tipo)}</td><td class="t-num">${t.cantidad}</td><td class="t-num">${soles(t.ingresos)}</td></tr>`).join('');

  if (typeof Chart === 'undefined') return;
  const css = getComputedStyle(document.documentElement);
  const v = n => css.getPropertyValue(n).trim();
  Chart.defaults.font.family = v('--fuente');
  Chart.defaults.color = v('--muted');

  graficos.tec?.destroy();
  graficos.tec = new Chart($('#g-tecnicos'), {
    type: 'bar',
    data: {
      labels: r.por_tecnico.map(x => x.tecnico),
      datasets: [{ label: 'Ingresos', data: r.por_tecnico.map(x => x.ingresos), backgroundColor: v('--accent'), borderRadius: 6, maxBarThickness: 36 }],
    },
    options: {
      indexAxis: 'y', maintainAspectRatio: false,
      plugins: { legend: { display: false }, tooltip: { callbacks: { label: c => soles(c.raw) } } },
      scales: { x: { grid: { color: v('--line') }, ticks: { callback: n => 'S/ ' + n } }, y: { grid: { display: false } } },
    },
  });

  graficos.est?.destroy();
  graficos.est = new Chart($('#g-estados'), {
    type: 'doughnut',
    data: {
      labels: catalogos.estados,
      datasets: [{
        data: catalogos.estados.map(e => r.por_estado[e] || 0),
        backgroundColor: ['--pendiente', '--proceso', '--terminado', '--entregado'].map(v),
        borderColor: v('--surface'), borderWidth: 3,
      }],
    },
    options: { maintainAspectRatio: false, cutout: '62%', plugins: { legend: { position: 'bottom' } } },
  });
}

/* ---------- Eventos generales ---------- */
document.addEventListener('click', e => {
  const vista = e.target.closest('[data-view]');
  if (vista) return mostrarVista(vista.dataset.view);
  const avanzar = e.target.closest('[data-avanzar]');
  if (avanzar) {
    const s = servicios.find(x => x.id === Number(avanzar.dataset.avanzar));
    const i = catalogos.estados.indexOf(s.estado);
    return cambiarEstado(s.id, catalogos.estados[i + 1]);
  }
  const borrar = e.target.closest('[data-eliminar]');
  if (borrar) return eliminar(Number(borrar.dataset.eliminar));
  if (e.target.closest('[data-imprimir]')) window.print();
});

document.addEventListener('change', e => {
  const sel = e.target.closest('[data-estado-id]');
  if (sel) cambiarEstado(Number(sel.dataset.estadoId), sel.value);
});

$('#filtro-texto').addEventListener('input', renderTabla);
$('#filtro-estado').addEventListener('change', renderTabla);

(async function iniciar() {
  $('#form-servicio').fecha.value = hoy();
  try {
    await cargarCatalogos();
    await cargarServicios();
  } catch (e) {
    aviso('No se pudo conectar con el servidor. Revisa que la celda del servidor siga en ejecución en Colab.', 'error');
  }
})();

Writing /content/ts_web/static/app.js


## 6. 🚀 Iniciar el servidor y abrir la página
Haz clic en el enlace que aparece. La página funciona mientras esta sesión de Colab esté activa.

In [6]:
import subprocess, sys, time
from google.colab import output
from google.colab.output import eval_js

subprocess.run(['pkill', '-f', 'ts_web/app.py'])  # detiene una ejecución anterior, si existe
time.sleep(1)

registro = open('/content/ts_web/servidor.log', 'w')
servidor = subprocess.Popen(
    [sys.executable, '/content/ts_web/app.py'],
    env={**os.environ, 'DB_PATH': DB_PATH, 'PORT': '5000'},
    stdout=registro, stderr=subprocess.STDOUT,
)
time.sleep(3)

if servidor.poll() is not None:
    print('❌ El servidor no arrancó. Detalle del error:')
    print(open('/content/ts_web/servidor.log').read())
else:
    print('✅ Servidor en ejecución')
    print('Enlace directo:', eval_js('google.colab.kernel.proxyPort(5000)'))
    output.serve_kernel_port_as_window(5000, anchor_text='👉 Abrir Technology Solutions en una pestaña nueva')

✅ Servidor en ejecución
Enlace directo: https://5000-m-s-usw1c5-1bo7qgflx104w-c.us-west1.prod.r.colab.dev
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## 7. (Opcional) Ver la página dentro de Colab

In [7]:
output.serve_kernel_port_as_iframe(5000, height=850)

<IPython.core.display.Javascript object>

## 8. Ver errores del servidor o detenerlo

In [8]:
print(open('/content/ts_web/servidor.log').read()[-3000:])

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Nov/2026 00:16:54] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:16:55] "GET /static/styles.css HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:16:55] "GET /static/app.js HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:16:55] "GET /api/catalogos HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:16:55] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [12/Nov/2026 00:16:56] "GET /api/servicios HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:17:05] "GET /api/resumen HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:17:11] "GET /api/resumen HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:18:59] "GET /api/resumen HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:19:20] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:19:21] "GET /static/styles.css HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 00:19:21] "GET /s

In [9]:
servidor.terminate()
print('🛑 Servidor detenido')

🛑 Servidor detenido


## 9. 💾 Guardar en GitHub
1. Menú **Archivo → Guardar una copia en GitHub**.
2. Elige el repositorio `van1403/TECHNOLOGY-SOLUTIONS`.
3. Escribe un mensaje de commit (ej. *"Sistema web de servicio técnico"*) y confirma.

Así no necesitas poner tokens en el código.